
# Tile2Net aerial pedestrian-infrastructure pipeline (inference-only)

Runs the full Tile2Net pipeline on a **Colab GPU runtime** and produces the
router-contract GeoJSON files for the accessibility-aware walking/biking
project (DC area).

**Before running:** `Runtime -> Change runtime type -> GPU (T4/L4/A100)`.

Pipeline: env check -> install tile2net (pinned commit) -> Boston example
(environment validation) -> DC test area -> DC demo area -> schema
inspection -> contract GeoJSON export -> preview maps -> OSM comparison
metrics -> payload zip download.

**Drop the payload into the repo:**
| payload path | repo path |
|---|---|
| `data/*.geojson` | `data/` |
| `ml_aerial_previews/*` | `ml/aerial/previews/` |
| `metrics/aerial_vs_osm.json` | `metrics/` |
| `parquet_debug/*` | anywhere (debugging only, not committed) |

Notes:
* `DEMO_BBOX` placeholder: ~1 km2 Logan Circle, DC. Replace in the config
  cell when the real demo bbox is set (format `[lat, lon, lat, lon]`).
* DC and VA imagery sources are supported; **Maryland is not** -- do not run
  demo areas there.
* Runtime on T4: ~15-25 min total, < 2 GB disk.


In [ ]:

import os
import subprocess
import torch

print("nvidia-smi:")
subprocess.run(["nvidia-smi"], check=False)
cuda_ok = torch.cuda.is_available()
print("torch.cuda.is_available():", cuda_ok, "| torch:", torch.__version__)
assert cuda_ok, (
    "No CUDA GPU. Tile2Net inference requires one. "
    "Select Runtime > Change runtime type > GPU (T4/L4/A100) and rerun."
)



## Config — edit `DEMO_BBOX` here


In [ ]:

import os

BASE = "/kaggle/working" if os.path.exists("/kaggle") else "/content"

# [lat, lon, lat, lon] — top-left then bottom-right
DEMO_BBOX = (38.9182, -77.0380, 38.9092, -77.0270)   # ~1 km2 Logan Circle, DC — REPLACE when real bbox exists
TEST_BBOX = (38.9162, -77.0340, 38.9142, -77.0310)   # small fast-validation box
DEMO_REGION = "washington-dc"  # for in_training_cities: washington-dc | boston | cambridge | nyc

TILE2NET_URL = "https://github.com/VIDA-NYU/tile2net.git"
TILE2NET_COMMIT = "c737a4e"  # pinned for reproducibility; keep in sync with SETUP.md

OUTPUT_ROOT = f"{BASE}/output"
DATA_DIR = f"{BASE}/data"
PREVIEW_DIR = f"{BASE}/ml_aerial_previews"
METRICS_DIR = f"{BASE}/metrics"
PQ_DIR = f"{BASE}/parquet_debug"

for d in (OUTPUT_ROOT, DATA_DIR, PREVIEW_DIR, METRICS_DIR, PQ_DIR):
    os.makedirs(d, exist_ok=True)

print("BASE =", BASE)



## Install tile2net (pinned)


In [ ]:

import subprocess

repo_dir = f"{BASE}/tile2net"
if not os.path.exists(f"{repo_dir}/.git"):
    subprocess.run(["git", "clone", TILE2NET_URL, repo_dir], check=True)
subprocess.run(["git", "-C", repo_dir, "checkout", TILE2NET_COMMIT], check=True)
subprocess.run(["git", "-C", repo_dir, "rev-parse", "--short", "HEAD"], check=True)


In [ ]:

import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", repo_dir], check=True)
print("tile2net installed")



## Pipeline helpers


In [ ]:

import json
import re
import subprocess as sp
import sys


def bbox_str(bb):
    return ",".join(str(v) for v in bb)


def generate(location, name, out_root, dump_percent=0):
    cmd = [
        sys.executable, "-m", "tile2net", "generate",
        "--location", location, "--name", name,
        "--output", out_root, "--dump_percent", str(dump_percent),
    ]
    p = sp.run(cmd, cwd=repo_dir, capture_output=True, text=True)
    print(p.stdout[-3000:])
    print(p.stderr[-3000:])
    m = re.search(r"INFO\s+Dumping to (\S+\.json)", p.stdout + "\n" + p.stderr)
    if not m:
        raise RuntimeError(f"generate failed for {name}; see logs above")
    city_info = m.group(1)
    print("city_info:", city_info)
    return city_info


def infer(city_info, dump_percent=0, eval_mode="test"):
    cmd = [
        sys.executable, "-m", "tile2net", "inference",
        "--local", "--eval", eval_mode,
        "--city_info", city_info,
        "--dump_percent", str(dump_percent),
    ]
    p = sp.run(cmd, cwd=repo_dir, capture_output=True, text=True)
    print(p.stdout[-6000:])
    if p.returncode != 0:
        print(p.stderr[-6000:])
        raise RuntimeError("inference failed; see stderr above")
    print("inference ok:", city_info)
    return city_info



## 1. Environment validation — Boston example (headless `examples/example.sh`)


In [ ]:

BOSTON_BBOX = "42.35555189953313,-71.07168915322092,42.35364837213307,-71.06437423368418"

info = generate(BOSTON_BBOX, "example", OUTPUT_ROOT, dump_percent=100)
infer(info, dump_percent=100)
print("Boston example done — environment validated")



## 2. DC test area (validation, small)


In [ ]:

info_test = generate(bbox_str(TEST_BBOX), "test_area", OUTPUT_ROOT, dump_percent=0)
infer(info_test, dump_percent=0)



## 3. DC demo area


In [ ]:

info_demo = generate(bbox_str(DEMO_BBOX), "demo", OUTPUT_ROOT, dump_percent=0)
infer(info_demo, dump_percent=0)



## 4. Inspect raw outputs (names + schemas — mirror of SETUP.md)


In [ ]:

import geopandas as gpd
import glob

for area, name in (("test_area", "test_area"), ("demo", "demo")):
    net = f"{OUTPUT_ROOT}/{area}/network/{name}-network.parquet"
    pol = f"{OUTPUT_ROOT}/{area}/polygons/{name}-polygons.parquet"
    print(f"--- {area} ---")
    if os.path.exists(net):
        g = gpd.read_parquet(net)
        print("network:", len(g), "| crs:", g.crs, "| geom:", g.geom_type.value_counts().to_dict())
        print("network f_type:", g["f_type"].value_counts().to_dict())
    if os.path.exists(pol):
        g = gpd.read_parquet(pol)
        print("polygons:", len(g), "| crs:", g.crs, "| geom:", g.geom_type.value_counts().to_dict())
        print("polygons f_type:", g["f_type"].value_counts().to_dict())

print("--- all output dirs ---")
for d in sorted(glob.glob(f"{OUTPUT_ROOT}/**/", recursive=True))[:60]:
    print(d)



## 5. Run the postprocessing scripts (embedded copies of `ml/aerial/`)

The three `ml/aerial/` scripts are embedded below (base64, injected from the
committed files at build time) so the notebook is self-contained — no GitHub
dependency. If you edit those scripts, rebuild the notebook or re-paste.


In [ ]:

import base64
import pathlib

script_b64 = {
    "convert_tile2net_output.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJDb252ZXJ0IHRpbGUybmV0IHZlY3RvciBvdXRwdXRzIHRvIHRoZSByb3V0ZXIgY29udHJhY3QgR2VvSlNPTi4KClJlYWRzIHRoZSBHZW9QYXJxdWV0IGZpbGVzIHRpbGUybmV0J3MgaW5mZXJlbmNlIHN0ZXAgd3JpdGVzOgogIDxwcm9qZWN0LWRpcj4vbmV0d29yay88cHJvamVjdC1uYW1lPi1uZXR3b3JrLnBhcnF1ZXQgICAoTGluZVN0cmluZ3MsIEVQU0c6NDMyNikKICA8cHJvamVjdC1kaXI+L3BvbHlnb25zLzxwcm9qZWN0LW5hbWU+LXBvbHlnb25zLnBhcnF1ZXQgKFBvbHlnb25zLCBFUFNHOjQzMjYpCgpXcml0ZXMgTGluZVN0cmluZyAvIFBvbHlnb24gR2VvSlNPTiBpbiBFUFNHOjQzMjYgd2l0aCBwcm9wZXJ0aWVzCiAgeyJzb3VyY2UiOiAibWwiLCAia2luZCI6ICJzaWRld2FsayJ8ImNyb3Nzd2FsayJ8ImZvb3RwYXRoIiwgImNvbmZpZGVuY2UiOiBudWxsfQoKdGlsZTJuZXQgZXhwb3NlcyBubyBwZXItZmVhdHVyZSBzb2Z0bWF4L2NvbmZpZGVuY2UgaW4gaXRzIHZlY3RvciBvdXRwdXRzLCBzbwpjb25maWRlbmNlIGlzIGFsd2F5cyBudWxsIChkb2N1bWVudGVkIGluIFNFVFVQLm1kOyBuZXZlciBpbnZlbnRlZCkuCgpFeGNsdWRlZCBmX3R5cGUgdmFsdWVzICh0aWxlMm5ldCBhcnRpZmFjdHMsIG5vdCBkZXRlY3RlZCBmZWF0dXJlcyk6CiAgbmV0d29yazogc2lkZXdhbGtfY29ubmVjdGlvbiwgbWVkaWFucwogIHBvbHlnb25zOiByb2FkCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IHBhdGhsaWIKaW1wb3J0IHN5cwoKaW1wb3J0IGdlb3BhbmRhcyBhcyBncGQKCktJTkRfTUFQID0gewogICAgInNpZGV3YWxrIjogInNpZGV3YWxrIiwKICAgICJjcm9zc3dhbGsiOiAiY3Jvc3N3YWxrIiwKICAgICJmb290cGF0aCI6ICJmb290cGF0aCIsCiAgICAjIHRpbGUybmV0IGFydGlmYWN0cyB0aGF0IGFyZSBub3QgZGV0ZWN0ZWQgZmVhdHVyZXM6CiAgICAic2lkZXdhbGtfY29ubmVjdGlvbiI6IE5vbmUsCiAgICAibWVkaWFucyI6IE5vbmUsCiAgICAicm9hZCI6IE5vbmUsCn0KCgpkZWYgX3RvXzQzMjYoZ2RmOiBncGQuR2VvRGF0YUZyYW1lKSAtPiBncGQuR2VvRGF0YUZyYW1lOgogICAgaWYgZ2RmLmNycyBpcyBOb25lOgogICAgICAgIHJldHVybiBnZGYuc2V0X2Nycyg0MzI2KQogICAgaWYgZ2RmLmNycy50b19lcHNnKCkgIT0gNDMyNjoKICAgICAgICByZXR1cm4gZ2RmLnRvX2Nycyg0MzI2KQogICAgcmV0dXJuIGdkZgoKCmRlZiBfbWFwX2tpbmRzKGdkZjogZ3BkLkdlb0RhdGFGcmFtZSwgbGF5ZXI6IHN0cikgLT4gZ3BkLkdlb0RhdGFGcmFtZToKICAgIGlmICJmX3R5cGUiIG5vdCBpbiBnZGYuY29sdW1uczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICBmIntsYXllcn0gcGFycXVldCBoYXMgbm8gZl90eXBlIGNvbHVtbjogY29sdW1ucz17bGlzdChnZGYuY29sdW1ucyl9IgogICAgICAgICkKICAgIGtpbmRzID0gZ2RmWyJmX3R5cGUiXS5tYXAoS0lORF9NQVApCiAgICB1bmtub3duID0gc29ydGVkKHNldChnZGZbImZfdHlwZSJdLnVuaXF1ZSgpKSAtIHNldChLSU5EX01BUCkpCiAgICBpZiB1bmtub3duOgogICAgICAgIHByaW50KGYiV0FSTklORzogdW5rbm93biBmX3R5cGUgdmFsdWVzIGV4Y2x1ZGVkOiB7dW5rbm93bn0iLCBmaWxlPXN5cy5zdGRlcnIpCiAgICBnZGYgPSBnZGZba2luZHMubm90bmEoKV0uY29weSgpCiAgICBnZGZbImtpbmQiXSA9IGtpbmRzW2tpbmRzLm5vdG5hKCldCiAgICByZXR1cm4gZ2RmCgoKZGVmIF9jb250cmFjdF9nZGYoZ2RmOiBncGQuR2VvRGF0YUZyYW1lKSAtPiBncGQuR2VvRGF0YUZyYW1lOgogICAgb3V0ID0gZ3BkLkdlb0RhdGFGcmFtZSgKICAgICAgICB7CiAgICAgICAgICAgICJzb3VyY2UiOiAibWwiLAogICAgICAgICAgICAia2luZCI6IGdkZlsia2luZCJdLnZhbHVlcywKICAgICAgICAgICAgImNvbmZpZGVuY2UiOiBbTm9uZV0gKiBsZW4oZ2RmKSwKICAgICAgICB9LAogICAgICAgIGdlb21ldHJ5PWdkZi5nZW9tZXRyeS52YWx1ZXMsCiAgICAgICAgY3JzPTQzMjYsCiAgICApCiAgICByZXR1cm4gb3V0CgoKZGVmIGNvbnZlcnRfcHJvamVjdCgKICAgICAgICBwcm9qZWN0X2Rpcjogc3RyIHwgcGF0aGxpYi5QYXRoLAogICAgICAgIHByb2plY3RfbmFtZTogc3RyLAogICAgICAgIGluZmVycmVkX291dDogc3RyIHwgcGF0aGxpYi5QYXRoLAogICAgICAgIHBvbHlnb25zX291dDogc3RyIHwgcGF0aGxpYi5QYXRoLAopIC0+IHR1cGxlW2ludCwgaW50XToKICAgICIiIlJldHVybnMgKG5fbGluZXNfd3JpdHRlbiwgbl9wb2x5Z29uc193cml0dGVuKS4iIiIKICAgIHByb2plY3RfZGlyID0gcGF0aGxpYi5QYXRoKHByb2plY3RfZGlyKQogICAgbmV0X3BhdGggPSBwcm9qZWN0X2RpciAvICJuZXR3b3JrIiAvIGYie3Byb2plY3RfbmFtZX0tbmV0d29yay5wYXJxdWV0IgogICAgcG9sX3BhdGggPSBwcm9qZWN0X2RpciAvICJwb2x5Z29ucyIgLyBmIntwcm9qZWN0X25hbWV9LXBvbHlnb25zLnBhcnF1ZXQiCgogICAgaWYgbm90IG5ldF9wYXRoLmV4aXN0cygpIGFuZCBub3QgcG9sX3BhdGguZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoCiAgICAgICAgICAgIGYiTmVpdGhlciB7bmV0X3BhdGh9IG5vciB7cG9sX3BhdGh9IGZvdW5kOyAiCiAgICAgICAgICAgIGYiY2hlY2sgLS1wcm9qZWN0LWRpci8tLXByb2plY3QtbmFtZS4iCiAgICAgICAgKQoKICAgIG5fbGluZXMgPSBuX3BvbHlzID0gMAoKICAgIGlmIG5ldF9wYXRoLmV4aXN0cygpOgogICAgICAgIG5ldCA9IF9tYXBfa2luZHMoX3RvXzQzMjYoZ3BkLnJlYWRfcGFycXVldChuZXRfcGF0aCkpLCAibmV0d29yayIpCiAgICAgICAgbGluZXMgPSBfY29udHJhY3RfZ2RmKG5ldCkKICAgICAgICBpZiBsZW4obGluZXMpOgogICAgICAgICAgICAjIHNwbGl0IE11bHRpTGluZVN0cmluZyBwYXJ0cyBzbyB0aGUgcm91dGVyIG9ubHkgc2VlcyBMaW5lU3RyaW5ncwogICAgICAgICAgICBsaW5lcyA9IGxpbmVzLmV4cGxvZGUoaW5kZXhfcGFydHM9RmFsc2UpLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgICAgICAgICAgbGluZXMgPSBsaW5lc1tsaW5lcy5nZW9tZXRyeS5nZW9tX3R5cGUgPT0gIkxpbmVTdHJpbmciXQogICAgICAgIGlmIGxlbihsaW5lcyk6CiAgICAgICAgICAgIGluZmVycmVkX291dCA9IHBhdGhsaWIuUGF0aChpbmZlcnJlZF9vdXQpCiAgICAgICAgICAgIGluZmVycmVkX291dC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgICAgICBsaW5lcy50b19maWxlKGluZmVycmVkX291dCwgZHJpdmVyPSJHZW9KU09OIikKICAgICAgICAgICAgbl9saW5lcyA9IGxlbihsaW5lcykKICAgICAgICBlbHNlOgogICAgICAgICAgICBwcmludCgiV0FSTklORzogbm8gbGluZSBmZWF0dXJlcyBpbiBuZXR3b3JrIHBhcnF1ZXQiLCBmaWxlPXN5cy5zdGRlcnIpCgogICAgaWYgcG9sX3BhdGguZXhpc3RzKCk6CiAgICAgICAgcG9sID0gX21hcF9raW5kcyhfdG9fNDMyNihncGQucmVhZF9wYXJxdWV0KHBvbF9wYXRoKSksICJwb2x5Z29ucyIpCiAgICAgICAgcG9seXMgPSBfY29udHJhY3RfZ2RmKHBvbCkKICAgICAgICBpZiBsZW4ocG9seXMpOgogICAgICAgICAgICBwb2x5Z29uc19vdXQgPSBwYXRobGliLlBhdGgocG9seWdvbnNfb3V0KQogICAgICAgICAgICBwb2x5Z29uc19vdXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICAgICAgcG9seXMudG9fZmlsZShwb2x5Z29uc19vdXQsIGRyaXZlcj0iR2VvSlNPTiIpCiAgICAgICAgICAgIG5fcG9seXMgPSBsZW4ocG9seXMpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcHJpbnQoIldBUk5JTkc6IG5vIHBvbHlnb24gZmVhdHVyZXMgaW4gcG9seWdvbnMgcGFycXVldCIsIGZpbGU9c3lzLnN0ZGVycikKCiAgICByZXR1cm4gbl9saW5lcywgbl9wb2x5cwoKCmRlZiBtYWluKCkgLT4gaW50OgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigKICAgICAgICBkZXNjcmlwdGlvbj1fX2RvY19fLCBmb3JtYXR0ZXJfY2xhc3M9YXJncGFyc2UuUmF3RGVzY3JpcHRpb25IZWxwRm9ybWF0dGVyCiAgICApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcHJvamVjdC1kaXIiLCByZXF1aXJlZD1UcnVlLAogICAgICAgICAgICAgICAgICAgIGhlbHA9InRpbGUybmV0IG91dHB1dCBkaXIgZm9yIG9uZSBwcm9qZWN0IChjb250YWlucyBuZXR3b3JrLyBhbmQgcG9seWdvbnMvKSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcHJvamVjdC1uYW1lIiwgcmVxdWlyZWQ9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJwcm9qZWN0IG5hbWUgdXNlZCB3aXRoIHRpbGUybmV0IGdlbmVyYXRlIC1uIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1pbmZlcnJlZCIsIHJlcXVpcmVkPVRydWUsCiAgICAgICAgICAgICAgICAgICAgaGVscD0ib3V0cHV0IEdlb0pTT04gcGF0aCBmb3IgdGhlIGNlbnRlcmxpbmUgbmV0d29yayIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcG9seWdvbnMiLCByZXF1aXJlZD1UcnVlLAogICAgICAgICAgICAgICAgICAgIGhlbHA9Im91dHB1dCBHZW9KU09OIHBhdGggZm9yIHRoZSBwb2x5Z29ucyIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgbl9saW5lcywgbl9wb2x5cyA9IGNvbnZlcnRfcHJvamVjdCgKICAgICAgICBhcmdzLnByb2plY3RfZGlyLCBhcmdzLnByb2plY3RfbmFtZSwgYXJncy5pbmZlcnJlZCwgYXJncy5wb2x5Z29ucwogICAgKQogICAgcHJpbnQoZiJ3cm90ZSB7bl9saW5lc30gbGluZSBmZWF0dXJlcyAtPiB7YXJncy5pbmZlcnJlZH0iKQogICAgcHJpbnQoZiJ3cm90ZSB7bl9wb2x5c30gcG9seWdvbiBmZWF0dXJlcyAtPiB7YXJncy5wb2x5Z29uc30iKQogICAgaWYgbl9saW5lcyA9PSAwIGFuZCBuX3BvbHlzID09IDA6CiAgICAgICAgcHJpbnQoIldBUk5JTkc6IGJvdGggb3V0cHV0cyBlbXB0eTsgY2hlY2sgdGlsZTJuZXQgaW5mZXJlbmNlIGxvZ3MiLCBmaWxlPXN5cy5zdGRlcnIpCiAgICAgICAgcmV0dXJuIDEKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQo=",
    "osm_metrics.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJDb21wYXJlIHRpbGUybmV0IG91dHB1dCBhZ2FpbnN0IE9wZW5TdHJlZXRNYXAgZm9yIHRoZSBkZW1vIGJib3ggKENQVSkuCgpDb21wdXRlczoKICAoYSkgc2hhcmUgb2YgT1NNIHNpZGV3YWxrL2Nyb3NzaW5nIGxlbmd0aCB3aXRoaW4gMyBtIG9mIHRpbGUybmV0IGNlbnRlcmxpbmVzCiAgICAgIChhIHJlY2FsbCBwcm94eSk7CiAgKGIpIHNoYXJlIG9mIHRpbGUybmV0IHNpZGV3YWxrL2Nyb3Nzd2FsayBsZW5ndGggd2l0aCBubyBPU00gc2lkZXdhbGsgb3IKICAgICAgZm9vdHdheSB3YXkgd2l0aGluIDUgbSAoY2FuZGlkYXRlIG5ldyBlZGdlcyBmb3IgdGhlIHJvdXRlcik7CiAgKGMpIGluX3RyYWluaW5nX2NpdGllczogd2hldGhlciB0aGUgZGVtbyBhcmVhIGlzIGluc2lkZSBhIERDIC8gQm9zdG9uIC8KICAgICAgQ2FtYnJpZGdlIC8gTllDIGFyZWEgd2hlcmUgdGlsZTJuZXQgd2FzIHRyYWluZWQgKHJlc3VsdHMgb3B0aW1pc3RpYwogICAgICB0aGVyZSkuCgpDYXZlYXRzIGRvY3VtZW50ZWQgaW4gdGhlIG91dHB1dCBKU09OOiBPU00gc2lkZXdhbGsgZGF0YSBpcyBpbmNvbXBsZXRlIGFuZCBpcwpub3QgZ3JvdW5kIHRydXRoOyBtZXRyaWNzIGFyZSBwcm94eSBlc3RpbWF0ZXMuCgpVc2FnZToKICBweXRob24gb3NtX21ldHJpY3MucHkgLS1pbmZlcnJlZCBkYXRhL2RlbW9fYXJlYV9pbmZlcnJlZC5nZW9qc29uIFwKICAgICAgLS1iYm94ICIzOC45MTYyLC03Ny4wMzQwLDM4LjkxNDIsLTc3LjAzMTAiIFwKICAgICAgLS1yZWdpb24gIndhc2hpbmd0b24tZGMiIC0tb3V0IG1ldHJpY3MvYWVyaWFsX3ZzX29zbS5qc29uCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGRhdGV0aW1lCmltcG9ydCBqc29uCmltcG9ydCBwYXRobGliCmltcG9ydCBzeXMKCmltcG9ydCBnZW9wYW5kYXMgYXMgZ3BkCmltcG9ydCBvc21ueCBhcyBveAppbXBvcnQgc2hhcGVseQoKQ1JPU1NXQUxLX0tJTkRTID0geyJjcm9zc3dhbGsifQpXQUxLX0tJTkRTID0geyJzaWRld2FsayIsICJmb290cGF0aCIsICJjcm9zc3dhbGsifQoKVFJBSU5JTkdfUkVHSU9OUyA9IHsKICAgICJ3YXNoaW5ndG9uLWRjIjogIkRDIiwKICAgICJib3N0b24iOiAiQm9zdG9uLCBNQSIsCiAgICAiY2FtYnJpZGdlIjogIkNhbWJyaWRnZSwgTUEiLAogICAgIm55YyI6ICJOZXcgWW9yaywgTlkiLAogICAgIm5ldy15b3JrLWNpdHkiOiAiTmV3IFlvcmssIE5ZIiwKICAgICJtYW5oYXR0YW4iOiAiTmV3IFlvcmssIE5ZIiwKfQoKCmRlZiBwcm9qZWN0X3RvX21ldHJpYyhnZGY6IGdwZC5HZW9EYXRhRnJhbWUgfCBncGQuR2VvU2VyaWVzLCBsb246IGZsb2F0KSAtPiBncGQuR2VvRGF0YUZyYW1lOgogICAgdXRtX2Vwc2cgPSBpbnQoaW50KChsb24gKyAxODApIC8vIDYpICsgMzI2MDEpICAjIHJvdWdoIFVUTSB6b25lIGZyb20gbG9uZ2l0dWRlCiAgICByZXR1cm4gZ2RmLnRvX2Nycyh1dG1fZXBzZykKCgpkZWYgb3NtX3NpZGV3YWxrX3dheXMoYmJveDogdHVwbGVbZmxvYXQsIGZsb2F0LCBmbG9hdCwgZmxvYXRdKSAtPiBncGQuR2VvRGF0YUZyYW1lOgogICAgIiIiYmJveCA9IChsYXRfdG9wLCBsb25fbGVmdCwgbGF0X2JvdHRvbSwgbG9uX3JpZ2h0KS4KCiAgICBSZXR1cm5zIHdheXMgdGFnZ2VkIGFzIHNpZGV3YWxrLWVxdWl2YWxlbnQgKGZvb3R3YXk9c2lkZXdhbGsgLyBiYXJlCiAgICBoaWdod2F5PWZvb3R3YXkpIG9yIGNyb3NzaW5ncyAoZm9vdHdheT1jcm9zc2luZyksIGFzIExpbmVTdHJpbmdzIG9ubHkuCiAgICAiIiIKICAgIGxhdF90b3AsIGxvbl9sZWZ0LCBsYXRfYm90dG9tLCBsb25fcmlnaHQgPSBiYm94CiAgICB3ZXN0LCBib3R0b20sIGVhc3QsIHRvcCA9IGxvbl9sZWZ0LCBsYXRfYm90dG9tLCBsb25fcmlnaHQsIGxhdF90b3AKICAgIGZvb3QgPSBveC5mZWF0dXJlc19mcm9tX2Jib3goCiAgICAgICAgYmJveD0od2VzdCwgYm90dG9tLCBlYXN0LCB0b3ApLAogICAgICAgIHRhZ3M9eyJoaWdod2F5IjogWyJmb290d2F5Il19LAogICAgKQogICAgZm9vdCA9IGZvb3QucmVzZXRfaW5kZXgoKQogICAgZWxlbV9jb2wgPSAiZWxlbWVudF90eXBlIiBpZiAiZWxlbWVudF90eXBlIiBpbiBmb290LmNvbHVtbnMgZWxzZSAiZWxlbWVudCIKICAgIGZvb3QgPSBmb290W2Zvb3RbZWxlbV9jb2xdID09ICJ3YXkiXQogICAgZm9vdCA9IGZvb3RbZm9vdC5nZW9tZXRyeS5nZW9tX3R5cGUuaXNpbihbIkxpbmVTdHJpbmciLCAiTXVsdGlMaW5lU3RyaW5nIl0pXQogICAgZm9vdHdheV9jb2wgPSBmb290LmdldCgiZm9vdHdheSIsICIiKS5hc3R5cGUoc3RyKQogICAgaXNfc2lkZXdhbGsgPSAoZm9vdHdheV9jb2wgPT0gInNpZGV3YWxrIikgfCAoCiAgICAgICAgICAgIChmb290d2F5X2NvbCA9PSAibmFuIikgJiAoZm9vdC5nZXQoImhpZ2h3YXkiLCAiIikuYXN0eXBlKHN0cikgPT0gImZvb3R3YXkiKQogICAgKQogICAgaXNfY3Jvc3NpbmcgPSBmb290d2F5X2NvbCA9PSAiY3Jvc3NpbmciCiAgICBvdXQgPSBmb290W2lzX3NpZGV3YWxrIHwgaXNfY3Jvc3NpbmddLmNvcHkoKQogICAgb3V0WyJvc21fa2luZCJdID0gWyJjcm9zc3dhbGsiIGlmIGMgZWxzZSAic2lkZXdhbGsiCiAgICAgICAgICAgICAgICAgICAgICAgZm9yIGMgaW4gaXNfY3Jvc3NpbmdbaXNfc2lkZXdhbGsgfCBpc19jcm9zc2luZ11dCiAgICBrZWVwID0gWyJvc21fa2luZCIsICJoaWdod2F5IiwgImZvb3R3YXkiLCAiY3Jvc3NpbmciLCAiZ2VvbWV0cnkiXQogICAgcmV0dXJuIGdwZC5HZW9EYXRhRnJhbWUob3V0W2tlZXBdLCBjcnM9b3V0LmNycykKCgpkZWYgZXZhbHVhdGUoaW5mZXJyZWRfcGF0aDogc3RyIHwgcGF0aGxpYi5QYXRoLCBiYm94LCByZWdpb246IHN0cikgLT4gZGljdDoKICAgIGxhdF90b3AsIGxvbl9sZWZ0LCBsYXRfYm90dG9tLCBsb25fcmlnaHQgPSBiYm94CiAgICBjZW50ZXJfbG9uID0gKGxvbl9sZWZ0ICsgbG9uX3JpZ2h0KSAvIDIuMAoKICAgIGluZmVycmVkID0gZ3BkLnJlYWRfZmlsZShpbmZlcnJlZF9wYXRoKQogICAgaWYgaW5mZXJyZWQuY3JzIGlzIE5vbmU6CiAgICAgICAgaW5mZXJyZWQgPSBpbmZlcnJlZC5zZXRfY3JzKDQzMjYpCiAgICBlbGlmIGluZmVycmVkLmNycy50b19lcHNnKCkgIT0gNDMyNjoKICAgICAgICBpbmZlcnJlZCA9IGluZmVycmVkLnRvX2Nycyg0MzI2KQoKICAgIG9zbSA9IG9zbV9zaWRld2Fsa193YXlzKGJib3gpCgogICAgVDJOID0gcHJvamVjdF90b19tZXRyaWMoaW5mZXJyZWQsIGNlbnRlcl9sb24pCiAgICBPU01QID0gcHJvamVjdF90b19tZXRyaWMob3NtLCBjZW50ZXJfbG9uKQoKICAgIHQybl93YWxrID0gVDJOW1QyTlsia2luZCJdLmlzaW4oV0FMS19LSU5EUyldCiAgICBvc21fd2FsayA9IE9TTVBbT1NNUFsib3NtX2tpbmQiXS5pc2luKFdBTEtfS0lORFMpXQogICAgdDJuX3VuaW9uID0gc2hhcGVseS51bmlvbl9hbGwodDJuX3dhbGsuZ2VvbWV0cnkudmFsdWVzKSBpZiBsZW4odDJuX3dhbGspIGVsc2UgTm9uZQogICAgb3NtX3VuaW9uID0gc2hhcGVseS51bmlvbl9hbGwob3NtX3dhbGsuZ2VvbWV0cnkudmFsdWVzKSBpZiBsZW4ob3NtX3dhbGspIGVsc2UgTm9uZQoKICAgIG06IGRpY3QgPSB7fQogICAgbVsib3NtX3NpZGV3YWxrX2Nyb3NzaW5nX2xlbmd0aF9tIl0gPSBmbG9hdChvc21fd2Fsay5nZW9tZXRyeS5sZW5ndGguc3VtKCkpIGlmIGxlbihvc21fd2FsaykgZWxzZSAwLjAKICAgIG1bInRpbGUybmV0X3NpZGV3YWxrX2Nyb3Nzd2Fsa19sZW5ndGhfbSJdID0gZmxvYXQodDJuX3dhbGsuZ2VvbWV0cnkubGVuZ3RoLnN1bSgpKSBpZiBsZW4odDJuX3dhbGspIGVsc2UgMC4wCgogICAgaWYgdDJuX3VuaW9uIGlzIG5vdCBOb25lIGFuZCBvc21fdW5pb24gaXMgbm90IE5vbmU6CiAgICAgICAgd2l0aGluMyA9IHNoYXBlbHkuaW50ZXJzZWN0aW9uKG9zbV91bmlvbiwgdDJuX3VuaW9uLmJ1ZmZlcigzLjApKQogICAgICAgIG1bIm9zbV9sZW5ndGhfd2l0aGluXzNtX29mX3RpbGUybmV0X20iXSA9IGZsb2F0KHNoYXBlbHkubGVuZ3RoKHdpdGhpbjMpKQogICAgICAgIGJleW9uZDUgPSBzaGFwZWx5LmRpZmZlcmVuY2UodDJuX3VuaW9uLCBvc21fdW5pb24uYnVmZmVyKDUuMCkpCiAgICAgICAgbVsidGlsZTJuZXRfbGVuZ3RoX3dpdGhvdXRfb3NtX3dpdGhpbl81bV9tIl0gPSBmbG9hdChzaGFwZWx5Lmxlbmd0aChiZXlvbmQ1KSkKICAgIGVsaWYgdDJuX3VuaW9uIGlzIG5vdCBOb25lOgogICAgICAgICMgbm90aGluZyBvbiB0aGUgT1NNIHNpZGUgLT4gYWxsIHRpbGUybmV0IGxlbmd0aCBpcyAibmV3IgogICAgICAgIG1bIm9zbV9sZW5ndGhfd2l0aGluXzNtX29mX3RpbGUybmV0X20iXSA9IDAuMAogICAgICAgIG1bInRpbGUybmV0X2xlbmd0aF93aXRob3V0X29zbV93aXRoaW5fNW1fbSJdID0gZmxvYXQoc2hhcGVseS5sZW5ndGgodDJuX3VuaW9uKSkKCiAgICBkZW5vbSA9IG0uZ2V0KCJvc21fc2lkZXdhbGtfY3Jvc3NpbmdfbGVuZ3RoX20iKSBvciAwLjAKICAgIG1bIm9zbV9yZWNhbGxfcHJveHlfc2hhcmUiXSA9ICgKICAgICAgICByb3VuZChtLmdldCgib3NtX2xlbmd0aF93aXRoaW5fM21fb2ZfdGlsZTJuZXRfbSIsIDAuMCkgLyBkZW5vbSwgNCkKICAgICAgICBpZiBkZW5vbSBlbHNlIE5vbmUKICAgICkKICAgIGRlbm9tMiA9IG0uZ2V0KCJ0aWxlMm5ldF9zaWRld2Fsa19jcm9zc3dhbGtfbGVuZ3RoX20iKSBvciAwLjAKICAgIG1bInRpbGUybmV0X25ld19lZGdlX3NoYXJlIl0gPSAoCiAgICAgICAgcm91bmQobS5nZXQoInRpbGUybmV0X2xlbmd0aF93aXRob3V0X29zbV93aXRoaW5fNW1fbSIsIDAuMCkgLyBkZW5vbTIsIDQpCiAgICAgICAgaWYgZGVub20yIGVsc2UgTm9uZQogICAgKQoKICAgIGluX3RyYWluaW5nID0gcmVnaW9uLmxvd2VyKCkuc3RyaXAoKSBpbiBUUkFJTklOR19SRUdJT05TCiAgICByZXR1cm4gewogICAgICAgICJnZW5lcmF0ZWRfdXRjIjogZGF0ZXRpbWUuZGF0ZXRpbWUubm93KGRhdGV0aW1lLnRpbWV6b25lLnV0YykuaXNvZm9ybWF0KHRpbWVzcGVjPSJzZWNvbmRzIiksCiAgICAgICAgInJlZ2lvbiI6IHJlZ2lvbiwKICAgICAgICAiZGVtb19iYm94IjogbGlzdChiYm94KSwKICAgICAgICAib3NtX3NpZGV3YWxrX3dheXMiOiBpbnQobGVuKG9zbV93YWxrKSksCiAgICAgICAgInRpbGUybmV0X2xpbmVzIjogaW50KGxlbih0Mm5fd2FsaykpLAogICAgICAgICJpbl90cmFpbmluZ19jaXRpZXMiOiBpbl90cmFpbmluZywKICAgICAgICAidHJhaW5pbmdfY2l0aWVzIjogc29ydGVkKHNldChUUkFJTklOR19SRUdJT05TLnZhbHVlcygpKSksCiAgICAgICAgKiptLAogICAgICAgICJjYXZlYXQiOiAoCiAgICAgICAgICAgICJPcGVuU3RyZWV0TWFwIHNpZGV3YWxrIGRhdGEgaXMgaW5jb21wbGV0ZSBhbmQgaXMgbm90IGdyb3VuZCB0cnV0aDsgIgogICAgICAgICAgICAidGhlc2UgYXJlIHByb3h5IG1ldHJpY3MsIG5vdCBhY2N1cmFjeSBtZWFzdXJlbWVudHMuIgogICAgICAgICksCiAgICB9CgoKZGVmIG1haW4oKSAtPiBpbnQ6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKAogICAgICAgIGRlc2NyaXB0aW9uPV9fZG9jX18sIGZvcm1hdHRlcl9jbGFzcz1hcmdwYXJzZS5SYXdEZXNjcmlwdGlvbkhlbHBGb3JtYXR0ZXIKICAgICkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1pbmZlcnJlZCIsIHJlcXVpcmVkPVRydWUsIGhlbHA9ImNvbnRyYWN0IGNlbnRlcmxpbmUgR2VvSlNPTiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYmJveCIsIHJlcXVpcmVkPVRydWUsIGhlbHA9ImxhdCxsb24sbGF0LGxvbiAodG9wLWxlZnQgdGhlbiBib3R0b20tcmlnaHQpIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1yZWdpb24iLCBkZWZhdWx0PSJ3YXNoaW5ndG9uLWRjIiwKICAgICAgICAgICAgICAgICAgICBoZWxwPWYib25lIG9mIHsnLCAnLmpvaW4oc29ydGVkKFRSQUlOSU5HX1JFR0lPTlMpKX0iKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW91dCIsIHJlcXVpcmVkPVRydWUsIGhlbHA9Im1ldHJpY3MgSlNPTiBvdXRwdXQgcGF0aCIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgYmJveCA9IHR1cGxlKGZsb2F0KHYpIGZvciB2IGluIGFyZ3MuYmJveC5zcGxpdCgiLCIpKQogICAgaWYgbGVuKGJib3gpICE9IDQ6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiLS1iYm94IG11c3QgYmUgZm91ciBjb21tYS1zZXBhcmF0ZWQgbnVtYmVycyBsYXQsbG9uLGxhdCxsb24iKQoKICAgIHRyeToKICAgICAgICBtZXRyaWNzID0gZXZhbHVhdGUoYXJncy5pbmZlcnJlZCwgYmJveCwgYXJncy5yZWdpb24pCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcHJpbnQoZiJFUlJPUjoge2V9IiwgZmlsZT1zeXMuc3RkZXJyKQogICAgICAgIHJldHVybiAxCgogICAgb3V0ID0gcGF0aGxpYi5QYXRoKGFyZ3Mub3V0KQogICAgb3V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBvdXQud3JpdGVfdGV4dChqc29uLmR1bXBzKG1ldHJpY3MsIGluZGVudD0yKSArICJcbiIpCiAgICBwcmludChqc29uLmR1bXBzKG1ldHJpY3MsIGluZGVudD0yKSkKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQo=",
    "make_previews.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJCdWlsZCBwcmV2aWV3IG1hcHMgKGZvbGl1bSBIVE1MICsgbWF0cGxvdGxpYiBQTkcpIGZyb20gY29udHJhY3QgR2VvSlNPTnMuCgpVc2FnZToKICBweXRob24gbWFrZV9wcmV2aWV3cy5weSAtLWluZmVycmVkIGRhdGEveF9pbmZlcnJlZC5nZW9qc29uIFwKICAgICAgLS1wb2x5Z29ucyBkYXRhL3hfcG9seWdvbnMuZ2VvanNvbiBcCiAgICAgIC0tYmJveCAibGF0LGxvbixsYXQsbG9uIiAtLW5hbWUgdGVzdF9hcmVhIC0tb3V0IG1sL2FlcmlhbC9wcmV2aWV3cwoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBwYXRobGliCmltcG9ydCB3YXJuaW5ncwoKaW1wb3J0IGZvbGl1bQppbXBvcnQgZ2VvcGFuZGFzIGFzIGdwZAppbXBvcnQgbWF0cGxvdGxpYgoKbWF0cGxvdGxpYi51c2UoIkFnZyIpCmltcG9ydCBtYXRwbG90bGliLnB5cGxvdCBhcyBwbHQKaW1wb3J0IHNoYXBlbHkKCktJTkRfQ09MT1JTID0gewogICAgInNpZGV3YWxrIjogIiMxZjc3YjQiLAogICAgImNyb3Nzd2FsayI6ICIjZmY3ZjBlIiwKICAgICJmb290cGF0aCI6ICIjMmNhMDJjIiwKfQoKCmRlZiBfbG9hZChwYXRoOiBzdHIgfCBwYXRobGliLlBhdGgsIGNycz00MzI2KSAtPiBncGQuR2VvRGF0YUZyYW1lOgogICAgZ2RmID0gZ3BkLnJlYWRfZmlsZShwYXRoKQogICAgaWYgZ2RmLmNycyBpcyBOb25lOgogICAgICAgIGdkZiA9IGdkZi5zZXRfY3JzKGNycykKICAgIGVsaWYgZ2RmLmNycy50b19lcHNnKCkgIT0gY3JzOgogICAgICAgIGdkZiA9IGdkZi50b19jcnMoY3JzKQogICAgcmV0dXJuIGdkZgoKCmRlZiBidWlsZCgKICAgICAgICBpbmZlcnJlZF9wYXRoOiBzdHIsCiAgICAgICAgcG9seWdvbnNfcGF0aDogc3RyIHwgTm9uZSwKICAgICAgICBiYm94OiB0dXBsZVtmbG9hdCwgZmxvYXQsIGZsb2F0LCBmbG9hdF0sCiAgICAgICAgbmFtZTogc3RyLAogICAgICAgIG91dF9kaXI6IHN0ciB8IHBhdGhsaWIuUGF0aCwKKSAtPiB0dXBsZVtwYXRobGliLlBhdGgsIHBhdGhsaWIuUGF0aF06CiAgICBvdXRfZGlyID0gcGF0aGxpYi5QYXRoKG91dF9kaXIpCiAgICBvdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBsYXRfdG9wLCBsb25fbGVmdCwgbGF0X2JvdHRvbSwgbG9uX3JpZ2h0ID0gYmJveAogICAgY2VudGVyID0gKChsYXRfdG9wICsgbGF0X2JvdHRvbSkgLyAyLjAsIChsb25fbGVmdCArIGxvbl9yaWdodCkgLyAyLjApCgogICAgZm1hcCA9IGZvbGl1bS5NYXAobG9jYXRpb249Y2VudGVyLCB6b29tX3N0YXJ0PTE2KQogICAgYmJveF9wb2x5ID0gZm9saXVtLlBvbHlnb24oCiAgICAgICAgbG9jYXRpb25zPVsobGF0X3RvcCwgbG9uX2xlZnQpLCAobGF0X3RvcCwgbG9uX3JpZ2h0KSwKICAgICAgICAgICAgICAgICAgIChsYXRfYm90dG9tLCBsb25fcmlnaHQpLCAobGF0X2JvdHRvbSwgbG9uX2xlZnQpXSwKICAgICAgICBjb2xvcj0iYmxhY2siLCB3ZWlnaHQ9MiwgZmlsbD1GYWxzZSwgdG9vbHRpcD0iYmJveCIsCiAgICApCiAgICBiYm94X3BvbHkuYWRkX3RvKGZtYXApCgogICAga2luZF9vZiA9IGxhbWJkYSBmZWF0dXJlOiAoKGZlYXR1cmUgb3Ige30pLmdldCgicHJvcGVydGllcyIpIG9yIHt9KS5nZXQoImtpbmQiKSAgIyBub3FhOiBFNzMxCgogICAgaWYgaW5mZXJyZWRfcGF0aCBhbmQgcGF0aGxpYi5QYXRoKGluZmVycmVkX3BhdGgpLmV4aXN0cygpOgogICAgICAgIGxpbmVzID0gX2xvYWQoaW5mZXJyZWRfcGF0aCkKICAgICAgICBzdHlsZSA9IGxhbWJkYSBmZWF0dXJlOiB7ICAjIG5vcWE6IEU3MzEKICAgICAgICAgICAgImNvbG9yIjogS0lORF9DT0xPUlMuZ2V0KGtpbmRfb2YoZmVhdHVyZSksICIjODg4ODg4IiksCiAgICAgICAgICAgICJ3ZWlnaHQiOiAzLAogICAgICAgIH0KICAgICAgICBmb2xpdW0uR2VvSnNvbihsaW5lcy5fX2dlb19pbnRlcmZhY2VfXywgc3R5bGVfZnVuY3Rpb249c3R5bGUsCiAgICAgICAgICAgICAgICAgICAgICAgbmFtZT0iY2VudGVybGluZXMiKS5hZGRfdG8oZm1hcCkKICAgIGlmIHBvbHlnb25zX3BhdGggYW5kIHBhdGhsaWIuUGF0aChwb2x5Z29uc19wYXRoKS5leGlzdHMoKToKICAgICAgICBwb2x5cyA9IF9sb2FkKHBvbHlnb25zX3BhdGgpCiAgICAgICAgcHN0eWxlID0gbGFtYmRhIGZlYXR1cmU6IHsgICMgbm9xYTogRTczMQogICAgICAgICAgICAiY29sb3IiOiBLSU5EX0NPTE9SUy5nZXQoa2luZF9vZihmZWF0dXJlKSwgIiM4ODg4ODgiKSwKICAgICAgICAgICAgIndlaWdodCI6IDEsICJmaWxsT3BhY2l0eSI6IDAuMjUsCiAgICAgICAgfQogICAgICAgIGZvbGl1bS5HZW9Kc29uKHBvbHlzLl9fZ2VvX2ludGVyZmFjZV9fLCBzdHlsZV9mdW5jdGlvbj1wc3R5bGUsCiAgICAgICAgICAgICAgICAgICAgICAgbmFtZT0icG9seWdvbnMiKS5hZGRfdG8oZm1hcCkKICAgIGZvbGl1bS5MYXllckNvbnRyb2woKS5hZGRfdG8oZm1hcCkKICAgIGZtYXAuZml0X2JvdW5kcyhbW2xhdF9ib3R0b20sIGxvbl9sZWZ0XSwgW2xhdF90b3AsIGxvbl9yaWdodF1dKQoKICAgIGh0bWxfcGF0aCA9IG91dF9kaXIgLyBmIntuYW1lfV9wcmV2aWV3Lmh0bWwiCiAgICBmbWFwLnNhdmUoaHRtbF9wYXRoKQoKICAgICMgbWF0cGxvdGxpYiBQTkcgKG5vIGJhc2VtYXAgZGVwZW5kZW5jeSkKICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oOCwgOCkpCiAgICBiYm94X2dkZiA9IGdwZC5HZW9EYXRhRnJhbWUoCiAgICAgICAgZ2VvbWV0cnk9W3NoYXBlbHkuYm94KGxvbl9sZWZ0LCBsYXRfYm90dG9tLCBsb25fcmlnaHQsIGxhdF90b3ApXSwgY3JzPTQzMjYKICAgICkKICAgIGJib3hfZ2RmLmJvdW5kYXJ5LnBsb3QoYXg9YXgsIGNvbG9yPSJibGFjayIsIGxpbmV3aWR0aD0xKQogICAgaWYgcG9seWdvbnNfcGF0aCBhbmQgcGF0aGxpYi5QYXRoKHBvbHlnb25zX3BhdGgpLmV4aXN0cygpOgogICAgICAgIHBvbHlzID0gX2xvYWQocG9seWdvbnNfcGF0aCkKICAgICAgICBmb3Iga2luZCwgc3ViIGluIHBvbHlzLmdyb3VwYnkoImtpbmQiKToKICAgICAgICAgICAgc3ViLmJvdW5kYXJ5LnBsb3QoYXg9YXgsIGNvbG9yPUtJTkRfQ09MT1JTLmdldChraW5kLCAiIzg4OCIpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBsaW5ld2lkdGg9MC40LCBsYWJlbD1mIntraW5kfSAocG9seSkiKQogICAgaWYgaW5mZXJyZWRfcGF0aCBhbmQgcGF0aGxpYi5QYXRoKGluZmVycmVkX3BhdGgpLmV4aXN0cygpOgogICAgICAgIGxpbmVzID0gX2xvYWQoaW5mZXJyZWRfcGF0aCkKICAgICAgICBmb3Iga2luZCwgc3ViIGluIGxpbmVzLmdyb3VwYnkoImtpbmQiKToKICAgICAgICAgICAgc3ViLnBsb3QoYXg9YXgsIGNvbG9yPUtJTkRfQ09MT1JTLmdldChraW5kLCAiIzg4OCIpLCBsaW5ld2lkdGg9MS4yLAogICAgICAgICAgICAgICAgICAgICBsYWJlbD1mIntraW5kfSAobGluZSkiKQogICAgYXgubGVnZW5kKGxvYz0ibG93ZXIgcmlnaHQiLCBmb250c2l6ZT03KQogICAgYXguc2V0X2F4aXNfb2ZmKCkKICAgIHBuZ19wYXRoID0gb3V0X2RpciAvIGYie25hbWV9X3ByZXZpZXcucG5nIgogICAgZmlnLnNhdmVmaWcocG5nX3BhdGgsIGRwaT0xNTAsIGJib3hfaW5jaGVzPSJ0aWdodCIpCiAgICBwbHQuY2xvc2UoZmlnKQoKICAgIHJldHVybiBodG1sX3BhdGgsIHBuZ19wYXRoCgoKZGVmIG1haW4oKSAtPiBpbnQ6CiAgICB3aXRoIHdhcm5pbmdzLmNhdGNoX3dhcm5pbmdzKCk6CiAgICAgICAgd2FybmluZ3Muc2ltcGxlZmlsdGVyKCJpZ25vcmUiKQogICAgICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoCiAgICAgICAgICAgIGRlc2NyaXB0aW9uPV9fZG9jX18sIGZvcm1hdHRlcl9jbGFzcz1hcmdwYXJzZS5SYXdEZXNjcmlwdGlvbkhlbHBGb3JtYXR0ZXIKICAgICAgICApCiAgICAgICAgYXAuYWRkX2FyZ3VtZW50KCItLWluZmVycmVkIiwgcmVxdWlyZWQ9VHJ1ZSkKICAgICAgICBhcC5hZGRfYXJndW1lbnQoIi0tcG9seWdvbnMiLCBkZWZhdWx0PU5vbmUpCiAgICAgICAgYXAuYWRkX2FyZ3VtZW50KCItLWJib3giLCByZXF1aXJlZD1UcnVlLCBoZWxwPSJsYXQsbG9uLGxhdCxsb24iKQogICAgICAgIGFwLmFkZF9hcmd1bWVudCgiLS1uYW1lIiwgcmVxdWlyZWQ9VHJ1ZSkKICAgICAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgcmVxdWlyZWQ9VHJ1ZSkKICAgICAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgICAgIGJib3ggPSB0dXBsZShmbG9hdCh2KSBmb3IgdiBpbiBhcmdzLmJib3guc3BsaXQoIiwiKSkKICAgICAgICBpZiBsZW4oYmJveCkgIT0gNDoKICAgICAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiLS1iYm94IG11c3QgYmUgbGF0LGxvbixsYXQsbG9uIikKICAgICAgICBodG1sX3BhdGgsIHBuZ19wYXRoID0gYnVpbGQoCiAgICAgICAgICAgIGFyZ3MuaW5mZXJyZWQsIGFyZ3MucG9seWdvbnMsIGJib3gsIGFyZ3MubmFtZSwgYXJncy5vdXQKICAgICAgICApCiAgICAgICAgcHJpbnQoZiJ3cm90ZSB7aHRtbF9wYXRofVxud3JvdGUge3BuZ19wYXRofSIpCiAgICAgICAgcmV0dXJuIDAKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpCg==",
}
for name, b in script_b64.items():
    pathlib.Path(BASE, name).write_bytes(base64.b64decode(b))
sys.path.insert(0, BASE)
import convert_tile2net_output
import make_previews
import osm_metrics
print("scripts written to", BASE)


In [ ]:

test_project = f"{OUTPUT_ROOT}/test_area"
demo_project = f"{OUTPUT_ROOT}/demo"

convert_tile2net_output.convert_project(
    test_project, "test_area",
    f"{DATA_DIR}/test_area_inferred.geojson",
    f"{DATA_DIR}/test_area_polygons.geojson",
)
convert_tile2net_output.convert_project(
    demo_project, "demo",
    f"{DATA_DIR}/demo_area_inferred.geojson",
    f"{DATA_DIR}/demo_area_polygons.geojson",
)
print("\ncontract files in", DATA_DIR)
for f in sorted(os.listdir(DATA_DIR)):
    print(" ", f)



## 6. Preview maps (folium HTML + PNG)


In [ ]:

make_previews.build(
    inferred_path=f"{DATA_DIR}/test_area_inferred.geojson",
    polygons_path=f"{DATA_DIR}/test_area_polygons.geojson",
    bbox=TEST_BBOX, name="test_area", out_dir=PREVIEW_DIR,
)
make_previews.build(
    inferred_path=f"{DATA_DIR}/demo_area_inferred.geojson",
    polygons_path=f"{DATA_DIR}/demo_area_polygons.geojson",
    bbox=DEMO_BBOX, name="demo_area", out_dir=PREVIEW_DIR,
)
print(os.listdir(PREVIEW_DIR))



## 7. OSM comparison metrics for the demo area

(a) share of OSM sidewalk/crossing length within 3 m of tile2net centerlines
(recall proxy); (b) share of tile2net sidewalk/crosswalk length with **no**
OSM sidewalk/footway within 5 m (candidate new edges); (c)
`in_training_cities` flag. OSM is incomplete and not ground truth — read the
caveat in the output JSON.


In [ ]:

import pathlib

metrics = osm_metrics.evaluate(
    f"{DATA_DIR}/demo_area_inferred.geojson", DEMO_BBOX, DEMO_REGION
)
print(json.dumps(metrics, indent=2))
pathlib.Path(f"{METRICS_DIR}/aerial_vs_osm.json").write_text(json.dumps(metrics, indent=2) + "\n")



## 8. Payload: parquet debug copies + zip download


In [ ]:

import shutil

for area, name in (("test_area", "test_area"), ("demo", "demo")):
    net_src = f"{OUTPUT_ROOT}/{area}/network/{name}-network.parquet"
    pol_src = f"{OUTPUT_ROOT}/{area}/polygons/{name}-polygons.parquet"
    for src in (net_src, pol_src):
        if os.path.exists(src):
            shutil.copy(src, f"{PQ_DIR}/{name}_{os.path.basename(src)}")
print(os.listdir(PQ_DIR))


In [ ]:

payload = f"{BASE}/tile2net_payload.zip"
p = sp.run(
    ["zip", "-r", payload, "data", "ml_aerial_previews", "metrics", "parquet_debug"],
    cwd=BASE, capture_output=True, text=True,
)
print(p.stdout[-2000:], p.stderr[-2000:])
print("payload:", payload, os.path.getsize(payload), "bytes")

try:
    from google.colab import files
    files.download(payload)
except Exception as e:
    print("auto-download unavailable:", e)
    print("Download tile2net_payload.zip from the file browser sidebar instead.")
